# R2 · Reranker + entities (Rathi)

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. It takes about 1–1.5 hours; keep the tab open.

Before running, add these in **Colab Secrets** (the key icon on the left) and switch on notebook access:
- `GITHUB_TOKEN`: a GitHub token with `repo` access, from the account that pushes (rathiabeth-sudo)
- `HF_TOKEN`: a Hugging Face token. It needs **write** access for step 6.
- `WANDB_KEY`: a Weights & Biases API key

Steps: 1 setup → 2 10-row checks → 3 fine-tune → 4 score all snippets → 5 entities → 6 save the model → 7 push the outputs to the `rathi/R2-reranker` branch.

In [ ]:
# 1. Setup. The repo is public, so cloning needs no token; GITHUB_TOKEN is only used to push at the end.
from google.colab import userdata
import os
REPO = '/content/twincheck'
if not os.path.exists(REPO):
    !git clone https://github.com/himaallu/Group-Project.git {REPO}
%cd {REPO}
!git fetch -q origin && git checkout -q rathi/R2-reranker && git pull -q origin rathi/R2-reranker
!git config user.name "Rathi"
!git config user.email "335395537+rathiabeth-sudo@users.noreply.github.com"

# Check the GitHub token now, not after an hour of training.
GH = userdata.get('GITHUB_TOKEN').strip()
!git remote set-url --push origin https://{GH}@github.com/himaallu/Group-Project.git
ok = !git ls-remote https://{GH}@github.com/himaallu/Group-Project.git HEAD
print('GitHub token OK' if ok and 'fatal' not in ' '.join(ok) else 'GITHUB_TOKEN rejected: make a new one with the "repo" box ticked')

# Colab already has torch, transformers, pandas, pyarrow and spaCy. Only add or upgrade what R2 needs.
!pip install -q -U wandb

# spaCy model: en_core_web_trf (CLAUDE.md); fall back to en_core_web_lg if trf cannot install on this Python.
import subprocess, sys
SPACY = 'en_core_web_trf'
if subprocess.run([sys.executable, '-m', 'spacy', 'download', SPACY]).returncode != 0 or \
        subprocess.run([sys.executable, '-c', f'import spacy; spacy.load("{SPACY}")']).returncode != 0:
    SPACY = 'en_core_web_lg'
    subprocess.run([sys.executable, '-m', 'spacy', 'download', SPACY], check=True)
print('spaCy model:', SPACY)

import wandb
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()
wandb.login(key=userdata.get('WANDB_KEY').strip(), relogin=True)   # error here = copy the key again from wandb.ai/authorize
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. 10-row checks (CLAUDE.md rule 3). Every command here must finish without errors before the full run.

In [ ]:
!python -m pytest tests/retrieval tests/data -q
!python -m src.retrieval.train_reranker --n-train-claims 5 --n-val-claims 5 --no-wandb     --out models/test10/reranker --metrics models/test10/eval.json --ckpt-dir /content/ckpt_test10
!python -m src.retrieval.rerank data/processed/en/multifc_val.jsonl --limit 10     --model models/test10/reranker --out features/test10/relevance_en.parquet
!python -m src.data.entities data/processed/en/multifc_val.jsonl --limit 10     --out features/test10/entities_en.jsonl --model {SPACY}
import pandas as pd, json
print(pd.read_parquet('features/test10/relevance_en.parquet').head(10))
print(open('features/test10/entities_en.jsonl', encoding='utf-8').readline()[:600])

## 3. Fine-tune the reranker: 20k pairs, 1 epoch, logged to W&B (~10–15 min on a T4)

In [ ]:
!python -m src.retrieval.train_reranker
print(open('results/R2/reranker_eval.json').read())

## 4. Score every (claim, snippet) pair in all English splits → `features/relevance_en.parquet` (~20 min)

In [ ]:
EN = 'data/processed/en'
FILES = ' '.join(f'{EN}/{d}_{s}.jsonl' for d in ('multifc', 'averitec') for s in ('train', 'val', 'test'))
!python -m src.retrieval.rerank {FILES} --model models/reranker_v1 --out features/relevance_en.parquet
rel = pd.read_parquet('features/relevance_en.parquet')
print(rel.describe())
print('share of snippets with relevance > 0.3:', round((rel.relevance > 0.3).mean(), 3))

## 5. spaCy entities → `features/entities_en.jsonl` (~30 min)

In [ ]:
!python -m src.data.entities {FILES} --out features/entities_en.jsonl --model {SPACY}
!ls -la features/

## 6. Save the fine-tuned model

The model is about 1.1 GB, which is more than GitHub's free Git LFS quota (1 GB for the whole repo). So it goes to **Hugging Face Hub** instead, and git keeps a small `models/reranker_v1/README.md` that points to it. Teammates load it with `Reranker("<hub id>")`.

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
HUB_ID = f"{api.whoami()['name']}/twincheck-reranker-v1"
api.create_repo(HUB_ID, exist_ok=True, private=False)
api.upload_folder(folder_path='models/reranker_v1', repo_id=HUB_ID,
                  commit_message='R2: bge-reranker-v2-m3 fine-tuned on MultiFC + AVeriTeC relevance pairs')
ev = json.load(open('results/R2/reranker_eval.json'))
open('models/reranker_v1/README.md', 'w').write(
    f"# reranker_v1 (R2)

Weights are on Hugging Face Hub: https://huggingface.co/{HUB_ID} "
    f"(too big for git).

Load: `from src.retrieval.rerank import Reranker; rr = Reranker('{HUB_ID}')`

"
    f"Base `{ev['base_model']}`, 1 epoch, {ev['n_train_pairs']} pairs, lr {ev['lr']}, seed {ev['seed']}. "
    f"Val AUROC {ev['val_zero_shot']['auroc']:.3f} -> {ev['val_fine_tuned']['auroc']:.3f}. W&B: {ev['wandb_url']}
")
print(open('models/reranker_v1/README.md').read())

## 7. Push the outputs to the `rathi/R2-reranker` branch

In [ ]:
!git config user.name "Rathi"
!git config user.email "335395537+rathiabeth-sudo@users.noreply.github.com"
!du -h features/relevance_en.parquet features/entities_en.jsonl
# GitHub refuses files over 100 MB; split entities by dataset if this fails.
!git add features/relevance_en.parquet features/entities_en.jsonl models/reranker_v1/README.md results/R2/reranker_eval.json
!git commit -q -m "[R2] Add relevance scores, entities and reranker eval from Colab run"
!git push -q origin rathi/R2-reranker && echo PUSHED

**Done.** Copy the output of steps 3–5 (or just the W&B link) back to Claude. Then **Runtime → Disconnect and delete runtime**.